# Den Fahrplan selbst durchrechnen (Connection Scan)

**Optionales Notebook zu Projekt 1 „Wo verliert der Nahverkehr gegen das Auto?“ (WiSe 2026/27)**

Ein fertiger Routenplaner wie MOTIS (siehe `reisezeiten_motis.ipynb`) beantwortet
Anfragen schnell und genau, aber der Fahrplan ist für ihn fest. Wenn Sie wissen wollen,
**wo der Engpass liegt und was eine einzelne Änderung bringt** (dichterer Takt, Rad als
Zubringer, kürzere Umstiege), ist es einfacher, den Fahrplan selbst in der Hand zu haben.

Dieses Notebook liest die offenen Fahrplandaten, berechnet mit dem *Connection Scan
Algorithm* (CSA) die früheste Ankunft und zeigt an einem Beispiel, wie Sie den Fahrplan
verändern. Es braucht nur `pandas` und `folium`, keinen Server.

**Daten:** `daten/dortmund-gtfs.zip` ist ein Zuschnitt des bundesweiten GTFS-Feeds von
[gtfs.de](https://gtfs.de) (Daten von DELFI e.V., CC BY 4.0) auf Dortmund und Umgebung.

**Vereinfachungen:** Fußwege sind Luftlinie mal 1,3 bei 4,3 km/h; zum ersten und vom
letzten Halt höchstens 800 m; Umsteigen dauert mindestens eine Minute. Verspätungen gibt
es nicht. MOTIS rechnet Fußwege auf dem echten Wegenetz und ist deshalb genauer.

In [1]:
import time

import pandas as pd

import gtfs_hilfen as h

if not h.GTFS_DATEI.exists():
    raise SystemExit("Das Datenpaket fehlt. Bitte zuerst im Wurzelverzeichnis ausführen: python daten/herunterladen.py")
t = time.time()
g = h.lade_gtfs()
print(f"geladen in {time.time() - t:.1f} s:", {name: len(df) for name, df in g.items()})

geladen in 1.2 s: {'agency': 27, 'routes': 677, 'trips': 62511, 'stops': 10639, 'stop_times': 1190160, 'calendar': 644, 'calendar_dates': 1311, 'feed_info': 1, 'attributions': 3}


## 1. Gilt der Fahrplan noch?

Der Feed von gtfs.de deckt nur etwa einen Monat ab. Für Tage außerhalb gibt es keine oder
falsche Verbindungen. `tag_waehlen` nimmt den nächsten Dienstag, der im Fahrplan liegt.

In [2]:
TAG = h.tag_waehlen(g, wochentag=1)        # 0 = Montag, 1 = Dienstag, ...
# from datetime import date; TAG = date(2026, 11, 24)     # oder ein fester Tag
h.pruefe_gueltigkeit(g, TAG)
wochentag = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"][TAG.weekday()]
print(f"Gerechnet wird für {wochentag}, {TAG:%d.%m.%Y}.")

Fahrplan gültig vom 07.11.2026 bis 07.12.2026.
Gerechnet wird für Dienstag, 17.11.2026.


## 2. Start und Ziel

Start und Ziel sind Koordinaten (Breite, Länge). Zu Fuß erreichbar sind alle Halte im
Umkreis von 800 m. Mit `h.halt_suchen(g, "Name")` finden Sie Halte und ihre Koordinaten.

In [3]:
START = (51.4862, 7.5050)   # Dortmund-Hörde
ZIEL = (51.4925, 7.4128)    # TU Dortmund, Campus Nord

zugang = h.halte_in_der_naehe(g, *START)
abgang = h.halte_in_der_naehe(g, *ZIEL)
print(len(zugang), "Halte am Start,", len(abgang), "Halte am Ziel, zum Beispiel:")
sorted({h.haltname(g, x) for x in abgang})[:8]

33 Halte am Start, 32 Halte am Ziel, zum Beispiel:


['DO-Joseph-von-Fraunhofer-Str.',
 'Dortmund Am Hartweg',
 'Dortmund Am Höhweg',
 'Dortmund Bundesanstalt f. Arb.',
 'Dortmund Campus Nord',
 'Dortmund Emil-Figge-Straße',
 'Dortmund Groß-Barop',
 'Dortmund Hauert']

## 3. Verbindungen eines Tages

Eine *Verbindung* (connection) ist die Fahrt eines Fahrzeugs von einem Halt zum nächsten,
mit Abfahrts- und Ankunftszeit. Der CSA braucht nur diese Liste, sortiert nach der
Abfahrtszeit.

In [4]:
t = time.time()
c = h.verbindungen(g, TAG, von="06:00", bis="12:00")
fuss = h.fusswege(g, max_meter=300)            # Umsteigen zwischen nahen Halten
print(f"{len(c)} Verbindungen in {time.time() - t:.1f} s")
c.head()

143222 Verbindungen in 3.4 s


,ab_halt,an_halt,ab,an,fahrt,einstieg_ok,ausstieg_ok,linie,typ
0,78384,92389,21600,21600,1046156,True,True,351,3
1,690802,680540,21600,21600,1537874,True,True,427,3
2,548311,457012,21600,21600,1749544,True,True,AST68,3
3,568484,562934,21600,21600,1752199,True,True,355,1
4,119684,223001,21600,21600,245283,True,True,14,3


## 4. Connection Scan: früheste Ankunft

Die Idee (Dibbelt, Pajor, Strasser, Wagner 2013): Gehe alle Verbindungen in der
Reihenfolge ihrer Abfahrt durch. Eine Verbindung ist benutzbar, wenn man schon in ihrer
Fahrt sitzt oder rechtzeitig an ihrem Abfahrtshalt ist. Dann ist man an ihrem
Ankunftshalt (und zu Fuß an den Halten in der Nähe) spätestens zu ihrer Ankunftszeit.

- `bereit[s]`: frühester Zeitpunkt, zu dem man an Halt `s` in ein Fahrzeug steigen kann
- `eingestiegen[f]`: wo man in Fahrt `f` eingestiegen ist (für die Rekonstruktion)
- Bei gleicher Ankunftszeit gewinnt die Reise mit weniger Fahrzeugen.

Anders als Dijkstra braucht der CSA keine Vorrangwarteschlange: eine Schleife über eine
sortierte Liste genügt. Dieselbe Funktion steht in `csa.py`.

In [5]:
from collections import defaultdict

UNENDLICH = float("inf")

def frueheste_ankunft(c, fuss, zugang, abgang, startzeit, direkt=UNENDLICH):
    bereit = defaultdict(lambda: UNENDLICH)
    fahrten_bis = defaultdict(int)            # benutzte Fahrzeuge bis dahin
    vorgaenger = {}                           # halt -> (einstieg, ausstieg, umstiegshalt)
    for halt, sekunden in zugang.items():
        bereit[halt] = startzeit + sekunden
    eingestiegen = {}                         # fahrt -> (Einstiegsverbindung, Fahrzeuge)
    beste, ziel_vorg, ziel_fahrten = startzeit + direkt, None, 0

    ab_halt, an_halt = c.ab_halt.tolist(), c.an_halt.tolist()
    ab, an, fahrt = c.ab.tolist(), c.an.tolist(), c.fahrt.tolist()
    ein_ok, aus_ok = c.einstieg_ok.tolist(), c.ausstieg_ok.tolist()

    for i in range(len(c)):
        if ab[i] > beste:
            break                             # später abfahren lohnt nicht mehr
        f, s0 = fahrt[i], ab_halt[i]
        hier_moeglich = ein_ok[i] and bereit[s0] <= ab[i]
        if f not in eingestiegen:
            if not hier_moeglich:
                continue                      # Fahrt (noch) nicht erreichbar
            eingestiegen[f] = (i, fahrten_bis[s0] + 1)
        elif hier_moeglich and fahrten_bis[s0] + 1 < eingestiegen[f][1]:
            eingestiegen[f] = (i, fahrten_bis[s0] + 1)   # hier einsteigen spart Umstiege
        if not aus_ok[i]:
            continue
        e, k = eingestiegen[f]
        s = an_halt[i]
        for nachbar, sekunden in fuss.get(s, [(s, h.UMSTIEGSZEIT)]):
            t = an[i] + sekunden
            if t < bereit[nachbar] or (t == bereit[nachbar] and k < fahrten_bis[nachbar]):
                bereit[nachbar], fahrten_bis[nachbar] = t, k
                vorgaenger[nachbar] = (e, i, s)
        if s in abgang:
            t = an[i] + abgang[s]
            if t < beste or (t == beste and k < ziel_fahrten):
                beste, ziel_vorg, ziel_fahrten = t, (e, i), k
    return beste, vorgaenger, ziel_vorg

In [6]:
import csa   # für die Rekonstruktion der Reise

STARTZEIT = 7 * 3600 + 30 * 60   # 7:30 Uhr, in Sekunden seit Mitternacht
t = time.time()
ankunft, vorgaenger, ziel_vorg = frueheste_ankunft(c, fuss, zugang, abgang, STARTZEIT, direkt=h.gehzeit(START, ZIEL))
reise = csa.rekonstruieren(c, vorgaenger, ziel_vorg)
print(f"Rechenzeit {1000 * (time.time() - t):.0f} ms; Ankunft am Ziel {h.uhrzeit(ankunft)}, "
      f"Reisezeit {(ankunft - STARTZEIT) / 60:.0f} min ab 07:30 Uhr")

def als_tabelle(reise):
    return pd.DataFrame([{"ab": h.uhrzeit(a["ab"]), "an": h.uhrzeit(a["an"]), "Linie": a["linie"],
                          "Verkehrsmittel": a["verkehrsmittel"], "von": h.haltname(g, a["von"]),
                          "nach": h.haltname(g, a["nach"])} for a in reise])
als_tabelle(reise)

Rechenzeit 130 ms; Ankunft am Ziel 08:09, Reisezeit 39 min ab 07:30 Uhr

,ab,an,Linie,Verkehrsmittel,von,nach
0,07:41,08:05,445,Bus,Dortmund Hörde Bf,Dortmund Universität


In [7]:
h.karte(g, [reise], start=START, ziel=ZIEL)

## 5. Am Fahrplan drehen: Was bringt eine einzelne Änderung?

Weil die Verbindungen eine einfache Tabelle sind, lassen sie sich leicht verändern.
Drei Beispiele für die Strecke oben:

1. **Dichterer Takt:** Jede Fahrt der Linie 445 bekommt eine zusätzliche Fahrt 10 Minuten später.
2. **Rad als Zubringer:** Zum ersten Halt geht es mit dem Rad (15 km/h, bis 3 km) statt zu Fuß.
3. **Späterer Start:** Wie stark hängt die Reisezeit vom Zufall der Abfahrtsminute ab?

In [8]:
def minuten(c, start=START, startzeit=STARTZEIT, **mehr):
    sekunden, reise = csa.reisezeit(g, c, fuss, start, ZIEL, startzeit, **mehr)
    return round(sekunden / 60), " → ".join(a["linie"] for a in reise) or "zu Fuß"

c_dicht = h.takt_verdichten(c, "445", versatz_min=10)
pd.DataFrame([
    ("Fahrplan wie er ist", *minuten(c)),
    ("Linie 445 doppelt so oft", *minuten(c_dicht)),
    ("mit dem Rad zum Halt", *minuten(c, zugang_meter=3000, zugang_tempo=4.2)),
], columns=["Variante", "Reisezeit (min)", "Linien"]).set_index("Variante")

,Reisezeit (min),Linien
Variante,,
Fahrplan wie er ist,39,445
Linie 445 doppelt so oft,39,445
mit dem Rad zum Halt,35,U41 → S1


In [9]:
# Reisezeit je nach Startminute: Wartezeit zählt mit
pd.DataFrame([{"Start": h.uhrzeit(STARTZEIT + m * 60), "Reisezeit (min)": minuten(c, startzeit=STARTZEIT + m * 60)[0],
               "Linien": minuten(c, startzeit=STARTZEIT + m * 60)[1]} for m in range(0, 31, 5)]).set_index("Start")

,Reisezeit (min),Linien
Start,,
07:30,39,445
07:35,44,445
07:40,39,445
07:45,35,RB53 → S1
07:50,30,RB53 → S1
07:55,40,RE57 → S2 → X13
08:00,35,RE57 → S2 → X13


## 6. Viele Startpunkte

Für eine Karte brauchen Sie viele Startpunkte. Hier ein kleines Beispiel mit einigen
Stadtteilen; für ein Raster über die ganze Stadt ersetzen Sie die Liste (siehe
`reisezeiten_motis.ipynb`). Eine Anfrage dauert etwa 0,3 Sekunden.

In [10]:
ORTE = {"Hörde": (51.4862, 7.5050), "Innenstadt (Hbf)": (51.5176, 7.4590), "Mengede": (51.5748, 7.3845),
        "Aplerbeck": (51.4926, 7.5570), "Lütgendortmund": (51.5003, 7.3470), "Eving": (51.5510, 7.4710),
        "Hombruch": (51.4770, 7.4390), "Brackel": (51.5240, 7.5460)}
zeilen = []
for name, ort in ORTE.items():
    zeilen.append({"Start": name, "wie er ist": minuten(c, start=ort)[0],
                   "445 doppelt so oft": minuten(c_dicht, start=ort)[0],
                   "Rad zum Halt": minuten(c, start=ort, zugang_meter=3000, zugang_tempo=4.2)[0],
                   "Linien (wie er ist)": minuten(c, start=ort)[1]})
pd.DataFrame(zeilen).set_index("Start")

,wie er ist,445 doppelt so oft,Rad zum Halt,Linien (wie er ist)
Start,,,,
Hörde,39,39,35,445
Innenstadt (Hbf),20,20,20,S1
Mengede,50,50,35,476 → RE3 → S1
Aplerbeck,49,49,49,RB59 → 445
Lütgendortmund,34,34,32,470 → S1
Eving,35,35,35,412 → U45 → S1
Hombruch,29,28,18,445
Brackel,43,43,43,U43 → U42 → E462


## 7. Ihr Projekt: Ideen

1. **Wo ist der Engpass?** Zerlegen Sie die Reisezeit in Weg zum Halt, Warten, Fahren und
   Umsteigen. Was dominiert in welchen Stadtteilen?
2. **Eine Maßnahme, viele Startpunkte:** Welche einzelne Änderung (Takt einer Linie,
   Rad als Zubringer, eine neue Direktverbindung) verbessert die Karte am meisten?
   Für wie viele Menschen? (Einwohnerzahlen im 100-m-Raster gibt es aus dem Zensus 2022.)
3. **Tageszeit und Takt:** Wie ändert sich das Bild abends oder am Sonntag?
4. **Annahmen prüfen:** Wie stark hängen die Ergebnisse von Gehgeschwindigkeit,
   Umwegfaktor und Umstiegszeit ab? Vergleichen Sie einzelne Verbindungen mit MOTIS.
5. **Mehr als die früheste Ankunft:** Der CSA oben optimiert nur die Ankunftszeit. Wie
   müsste man ihn erweitern, um auch die Zahl der Umstiege zu berücksichtigen?

Bitte nennen Sie auf Ihrem Poster die Datenquelle: Fahrplandaten gtfs.de / DELFI e.V. (CC BY 4.0).